# Environmental Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the eleven-question earthquake walkthrough in [1.5-pandas-environmental-exercises.ipynb](1.5-pandas-environmental-exercises.ipynb).
:::

## Exercise 1: Earthquake data analysis

```{figure} _static/usgs-2014-south-napa-earthquake.jpg
---
name: south-napa-surface-rupture
width: 800px
alt: A dirt track through a vineyard, broken along its length by a continuous ridge of pushed-up soil following the fault trace
---
Surface rupture from the magnitude 6.0 South Napa earthquake of 24 August 2014 — the kind of event a row in this catalog stands for. The continuous "mole track" running parallel to the strike of the fault indicates some east–west compression on top of the <a href="https://www.usgs.gov/faqs/what-fault-and-what-are-different-types">right-lateral faulting</a>; taken near Buhman Road. Photo <a href="https://www.usgs.gov/media/images/2014-south-napa-ca-m6-earthquake-august-24-6">USGS</a>, public domain.
```

This exercise reviews the `pandas` fundamentals of this subchapter on one real catalog, covering how to

* open csv files
* manipulate dataframe indexes
* parse date columns
* examine basic dataframe statistics
* manipulate text columns and extract values
* plot dataframe contents using
  * bar charts
  * histograms
  * scatter plots

The data is a snapshot of the [USGS Earthquakes Database](https://earthquake.usgs.gov/earthquakes/search/): every event recorded worldwide in 2014, 120 108 rows.

You do not need to download the file yourself. The pre-supplied cell below fetches and caches it with the help of [pooch](https://www.fatiando.org/pooch/latest/) — there is no need to read pooch's documentation unless you want to — and stores the path to the file in the variable `datafile`.


In [ ]:
# Pre-supplied: download and cache the earthquake data.
import pooch

datafile = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/usgs_earthquakes_2014.csv",
    known_hash="sha256:84d455fb96dc8f782fba4b5fbe56cb8970cab678f07c766fcba1b1c4674de1b1",
    fname="usgs_earthquakes_2014.csv",
    path=pooch.os_cache("mlees"),
)

**Q1) First, import `numpy`, `pandas` and `matplotlib`, and — optionally — set the display options.**

Hint: display options are documented [at this link](https://pandas.pydata.org/pandas-docs/stable/user_guide/options.html).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

**Q2) Use pandas' `read_csv` function directly on `datafile` to open it as a `DataFrame`.**

Display the first few rows with `.head()`, and the column names, dtypes and non-null counts with
`.info()`. Check these tutorials if you have doubts about what the two functions do:
[`.head()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html)
and
[`.info()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.info.html?highlight=info#pandas.DataFrame.info).

`.info()` will show you that the two date columns, `time` and `updated`, came in as plain strings
rather than as dates. They were not parsed automatically. What can you do about that?

In [ ]:
df = pd.read_csv(datafile)
print(df.shape)
print(df.head())
df.info()

# the time and updated columns came in as plain strings, not dates: pandas does not guess

**Q3) Re-read the data in such a way that both date columns are identified as dates and the earthquake ID is used as the index.**

Check with `.head()` and `.info()` that it worked: `time` and `updated` should now be
`datetime64` columns, and `id` should be the index rather than a column.

Hint: the documentation for `read_csv` is [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html?highlight=read_csv).

In [ ]:
df = pd.read_csv(datafile, parse_dates=["time", "updated"])
df = df.set_index("id")
print(df.head())
df.info()

**Q4) Use `describe` to get the basic statistics of all the columns.**

Hint: the documentation of `describe` is [at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html).

In [ ]:
print(df.describe())

**Q5) Use `nlargest` to get the top 20 earthquakes by magnitude.**

Hint: the documentation of `nlargest` is [at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html).

In [ ]:
print(df.nlargest(20, "mag")[["place", "mag", "depth"]])

**Q6) Extract the state or country using pandas' text data functions, and add it as a new column to the `DataFrame`.**

Examine the column titled `place`: it holds strings like `"26km S of Redoubt Volcano, Alaska"`, so
it carries both the local description and the state or country. Pull the second part out and store
it in a new column called `country`.

Hint 1: the documentation for pandas' text data functions is [here](https://pandas.pydata.org/pandas-docs/stable/text.html).

Hint 2: you will use `.split()` to extract the country names. The documentation of this function
can be found [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html?highlight=split#pandas.Series.str.split).

In [ ]:
country = df["place"].str.split(",", expand=True, n=1)
df["country"] = country[1]
print(df[["place", "country"]].head())

**Q7) Display each unique value from the new `country` column.**

Hint: you may use the `unique` function documented [at this link](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.unique.html).

You should see an array of a couple of hundred country and state names. A plain split leaves the
space that followed the comma, so they come out as `array([' Alaska', ' Nevada', ...])` — worth
removing before you use them as labels.

In [ ]:
print(df["country"].unique()[:12])
print("distinct values:", df["country"].nunique())

# a plain split leaves the space that followed the comma, hence ' Alaska' rather than 'Alaska'

**Q8) Create a filtered dataset that only has earthquakes larger than magnitude 4.**

Hint: print the table to see the name of the column containing the earthquake magnitude.

Check out this [link](https://www.geeksforgeeks.org/ways-to-filter-pandas-dataframe-by-column-values/)
to find examples of filtering by column values.

In [ ]:
df_filt = df[df["mag"] > 4]
print(df_filt.shape)

**Q9) Using the filtered dataset (magnitude > 4), count the number of earthquakes whose magnitudes are above 4 [Num 1], and count the number of earthquakes in each country or state [Num 2]. Make a bar chart of Num 2 for the top 5 locations with the most earthquakes.**

Hint 1: to get Num 1, pandas has a `count` function documented
[at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html).

Hint 2: check out the
[`value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
function to get Num 2.

Then convert the first five rows of Num 2 into a `DataFrame` with two columns, `country` (text)
and `earthquake_num` (number), and recreate the bar chart below.

```{figure} _static/1.5-target-top5-bar.png
---
name: target-top5-bar
width: 700px
alt: A bar chart of the number of magnitude-above-4 earthquakes in the five countries with the most of them
---
The five locations with the most magnitude-above-4 events in 2014.
```

In [ ]:
num1 = df_filt["mag"].count()
print("events above magnitude 4:", num1)

num2 = df_filt["country"].value_counts()
print(num2.head())

top5_df = pd.DataFrame({"country": list(num2.index[:5]),
                        "earthquake_num": list(num2.values[:5])})
print(top5_df)

fig, ax = plt.subplots(figsize=(7, 3.2))
# x and y name the columns to plot; rot=0 keeps the labels horizontal
top5_df.plot(kind="bar", x="country", y="earthquake_num", rot=0, ax=ax)
ax.set_ylabel("number of earthquakes")
plt.tight_layout()
plt.show()

**Q10) Make a histogram for the distribution of the earthquakes' magnitudes.**

Hint: pandas has a histogram function documented
[at this link](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.hist.html) and
matplotlib has one documented
[at this link](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.hist.html).

```{figure} _static/1.5-target-magnitude-hist.png
---
name: target-magnitude-hist
width: 700px
alt: A histogram of earthquake magnitude with a dashed grid
---
The distribution of magnitude across the whole catalog.
```

Then redraw it with a logarithmic scale on the y-axis, which brings the rare large events into
view alongside the many small ones. Hint: here you can find a
[tutorial](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.yscale.html) for how to
change an axis scale.

```{figure} _static/1.5-target-magnitude-hist-log.png
---
name: target-magnitude-hist-log
width: 700px
alt: The same histogram of earthquake magnitude with a logarithmic y-axis
---
The same histogram with a logarithmic count axis.
```

Finally, make one histogram for the filtered dataset and one for the unfiltered dataset, side by
side.

```{figure} _static/1.5-target-hist-filtered-unfiltered.png
---
name: target-hist-filtered-unfiltered
width: 100%
alt: Two histograms of earthquake magnitude side by side, one for events above magnitude 4 and one for the whole catalog
---
The filtered and unfiltered magnitude distributions, drawn on their own count axes.
```

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
df["mag"].plot(kind="hist", bins=20, ax=ax)
ax.set_xlabel("magnitude")
ax.set_ylabel("number")
ax.grid(alpha=0.2, color="b", linestyle="--")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.2))
df["mag"].plot(kind="hist", bins=20, ax=ax)
ax.set_xlabel("magnitude")
ax.set_ylabel("number")
ax.grid(alpha=0.2, color="b", linestyle="--")
ax.set_yscale("log")          # or plot(kind="hist", logy=True), as in the lecture
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
df_filt["mag"].plot(kind="hist", bins=20, color="#1f77b4", ax=axes[0])
axes[0].set_title("filtered")
df["mag"].plot(kind="hist", bins=20, color="#ff7f0e", ax=axes[1])
axes[1].set_title("unfiltered")
for ax in axes:
    ax.set_xlabel("magnitude")
    ax.set_ylabel("number")
    ax.grid(alpha=0.2, color="b", linestyle="--")
plt.tight_layout()
plt.show()

**Q11) Visualise the locations of earthquakes by making a scatterplot of their latitude and longitude.**

Make a figure with two panels: one for the filtered dataset and one for the unfiltered one, with
the points coloured by magnitude on a shared colour range.

Hint: consider reading the documentation for
[`plt.scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html) to make
the scatter plot and that of
[`plt.colorbar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.colorbar.html) to
colour the points by magnitude. Pass the same `vmin` and `vmax` to both scatter calls, so that a
colour stands for the same magnitude in both panels.

```{figure} _static/1.5-target-quake-scatter.png
---
name: target-quake-scatter
width: 100%
alt: Two scatter plots of earthquake longitude against latitude coloured by magnitude, one filtered to magnitude above 4 and one unfiltered
---
Every event in the catalog placed by longitude and latitude, filtered and unfiltered.
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, events, title in zip(axes, [df_filt, df], ["filtered", "unfiltered"]):
    sc = ax.scatter(events["longitude"], events["latitude"], c=events["mag"], s=4,
                    cmap="Reds", vmin=0, vmax=8)      # one colour range for both panels
    fig.colorbar(sc, ax=ax, label="magnitude")
    ax.set_xlabel("longitude (°E)")
    ax.set_ylabel("latitude (°N)")
    ax.set_title(title)
plt.tight_layout()
plt.show()

# Yes. Both panels trace the plate boundaries, but the unfiltered one also shows where the
# instruments are: dense pale clusters over California, Alaska, Oklahoma and Japan are small
# events that only a close network can record. The filtered map is closer to a map of
# tectonics; the unfiltered one is partly a map of seismometer coverage.